In [ ]:
import numpy as np
import pandas as pd
import pandas.api.types

import kaggle_metric_utilities


class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    '''
    Mean KL divergence between each predicted distribution row and the true one. Lower is better.

    Each row of the solution is a probability distribution over the columns (a pie that sums to 1).
    The submission must carry the same columns; each row is clipped to a small positive floor and
    renormalised, so it is scored on the shape of what it predicts. A missing row or column is an
    error shown to the participant.

    Examples
    --------
    >>> import pandas as pd
    >>> sol = pd.DataFrame({'year': [1, 2], 'a': [0.5, 0.2], 'b': [0.5, 0.8]})
    >>> score(sol.copy(), sol.copy(), row_id_column_name='year')
    0.0
    >>> uni = pd.DataFrame({'year': [1, 2], 'a': [0.5, 0.5], 'b': [0.5, 0.5]})
    >>> round(score(sol.copy(), uni, row_id_column_name='year'), 4)
    0.0964
    '''
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError(f'Submission is missing the row id column {row_id_column_name!r}')
    sol = solution.set_index(row_id_column_name).sort_index()
    # Kaggle attaches a Usage column (Public/Private) to every solution; it is not part of the pie
    sol = sol[[c for c in sol.columns if c != 'Usage' and pandas.api.types.is_numeric_dtype(sol[c])]]
    sub = submission.set_index(row_id_column_name)
    missing_rows = [i for i in sol.index if i not in sub.index]
    if missing_rows:
        raise ParticipantVisibleError(f'Submission is missing {len(missing_rows)} rows, e.g. {missing_rows[:3]}')
    missing_cols = [c for c in sol.columns if c not in sub.columns]
    if missing_cols:
        raise ParticipantVisibleError(f'Submission is missing {len(missing_cols)} columns, e.g. {missing_cols[:3]}')
    sub = sub.reindex(sol.index)[sol.columns]
    if not pandas.api.types.is_numeric_dtype(sub.values):
        bad = {c: str(sub[c].dtype) for c in sub.columns if not pandas.api.types.is_numeric_dtype(sub[c])}
        raise ParticipantVisibleError(f'Invalid submission data types found: {bad}')
    P = np.clip(sub.to_numpy(dtype=float), 1e-9, None)
    if not np.all(np.isfinite(P)):
        raise ParticipantVisibleError('Submission contains non-finite values')
    P = P / P.sum(axis=1, keepdims=True)
    Q = np.clip(sol.to_numpy(dtype=float), 0.0, None)
    Q = Q / Q.sum(axis=1, keepdims=True)
    kl = (Q * (np.log(np.maximum(Q, 1e-12)) - np.log(P))).sum(axis=1)
    return float(np.mean(kl))
